# SACP + GeoCP: Full HSI Experiments (Checkpointed)

**Method**: Standard CP vs SACP vs SACP+GeoCP on 5 HSI datasets.

**Protocol**: 10 seeds × 5 datasets, 3D-CNN trained from scratch, APS scoring, 5-fold CV for bandwidth.

**Robustness features**:
- Every seed's full results are pickled to Drive immediately after completion — **safe against runtime crashes**.
- Re-running the main loop skips already-completed (dataset, seed) combos — **safe to resume**.
- All aggregated outputs (JSON, CSV, LaTeX table, figures) are exported to Drive.

**Outputs** (to `MyDrive/sacp_geocp/`):
- `checkpoints/{ds}_seed{n}.pkl` — full per-seed data (probs, prediction sets, local thresholds)
- `results/summary.json` — aggregated mean±std per dataset
- `results/per_seed.csv` — long-format table for all runs
- `results/results_table.tex` — LaTeX-ready table
- `results/stats.json` — paired t-test / Wilcoxon results
- `figures/fig_all_datasets_results.png` — 5×4 classification map visualization

Run cells in order. If the runtime dies, start from the beginning; cached cells will finish instantly.

## 1. Setup & GPU check

In [1]:
# Install minimal deps (most are pre-installed on Colab)
!pip install -q scikit-learn scipy matplotlib

import torch, sys, os, time, gc
print(f'Python : {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'GPU    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Python : 3.12.13
PyTorch: 2.10.0+cu128
GPU    : True
Device : Tesla T4
Memory : 15.6 GB


## 2. Mount Drive & set up working directory

Results, checkpoints, and figures all go to `MyDrive/sacp_geocp/`. Set `USE_DRIVE = False` to fall back to ephemeral local storage.

In [2]:
USE_DRIVE = True  # set False to use local /content only (lost when runtime dies!)

WORK_DIR = '/content/sacp_geocp'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        WORK_DIR = '/content/drive/MyDrive/sacp_geocp'
        print(f'Using Drive: {WORK_DIR}')
    except Exception as e:
        print(f'Drive mount failed ({e}), falling back to local {WORK_DIR}')

DATA_DIR     = f'{WORK_DIR}/datasets'
CKPT_DIR     = f'{WORK_DIR}/checkpoints'
RESULTS_DIR  = f'{WORK_DIR}/results'
FIG_DIR      = f'{WORK_DIR}/figures'
for d in [WORK_DIR, DATA_DIR, CKPT_DIR, RESULTS_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'\nWORK_DIR    : {WORK_DIR}')
print(f'DATA_DIR    : {DATA_DIR}')
print(f'CKPT_DIR    : {CKPT_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'FIG_DIR     : {FIG_DIR}')


Mounted at /content/drive
Using Drive: /content/drive/MyDrive/sacp_geocp

WORK_DIR    : /content/drive/MyDrive/sacp_geocp
DATA_DIR    : /content/drive/MyDrive/sacp_geocp/datasets
CKPT_DIR    : /content/drive/MyDrive/sacp_geocp/checkpoints
RESULTS_DIR : /content/drive/MyDrive/sacp_geocp/results
FIG_DIR     : /content/drive/MyDrive/sacp_geocp/figures


## 3. Download HSI datasets (idempotent)

Downloads 5 datasets (~500 MB total). Skips files that are already present.

In [3]:
dataset_urls = {
    'ip': [('Indian_pines_corrected.mat', 'https://www.ehu.eus/ccwintco/uploads/6/67/Indian_pines_corrected.mat'),
           ('Indian_pines_gt.mat',        'https://www.ehu.eus/ccwintco/uploads/c/c4/Indian_pines_gt.mat')],
    'pu': [('PaviaU.mat',    'https://www.ehu.eus/ccwintco/uploads/e/ee/PaviaU.mat'),
           ('PaviaU_gt.mat', 'https://www.ehu.eus/ccwintco/uploads/5/50/PaviaU_gt.mat')],
    'sa': [('Salinas_corrected.mat', 'https://www.ehu.eus/ccwintco/uploads/a/a3/Salinas_corrected.mat'),
           ('Salinas_gt.mat',        'https://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat')],
    'ksc': [('KSC.mat',    'https://www.ehu.eus/ccwintco/uploads/2/26/KSC.mat'),
            ('KSC_gt.mat', 'https://www.ehu.eus/ccwintco/uploads/a/a6/KSC_gt.mat')],
    'botswana': [('Botswana.mat',    'https://www.ehu.eus/ccwintco/uploads/7/72/Botswana.mat'),
                 ('Botswana_gt.mat', 'https://www.ehu.eus/ccwintco/uploads/5/58/Botswana_gt.mat')],
}

for ds, files in dataset_urls.items():
    os.makedirs(f'{DATA_DIR}/{ds}', exist_ok=True)
    for fname, url in files:
        path = f'{DATA_DIR}/{ds}/{fname}'
        if os.path.exists(path) and os.path.getsize(path) > 10000:
            print(f'  [cached] {ds}/{fname}  ({os.path.getsize(path)//1024} KB)')
        else:
            print(f'  [DL]     {ds}/{fname}', end=' ... ', flush=True)
            os.system(f'wget -q -O "{path}" "{url}"')
            print(f'done ({os.path.getsize(path)//1024} KB)')
print('\nAll datasets ready')


  [cached] ip/Indian_pines_corrected.mat  (5813 KB)
  [DL]     ip/Indian_pines_gt.mat ... done (1 KB)
  [DL]     pu/PaviaU.mat ... done (33991 KB)
  [DL]     pu/PaviaU_gt.mat ... done (10 KB)
  [DL]     sa/Salinas_corrected.mat ... done (25930 KB)
  [DL]     sa/Salinas_gt.mat ... done (4 KB)
  [DL]     ksc/KSC.mat ... done (55492 KB)
  [DL]     ksc/KSC_gt.mat ... done (3 KB)
  [DL]     botswana/Botswana.mat ... done (77061 KB)
  [DL]     botswana/Botswana_gt.mat ... done (3 KB)

All datasets ready


## 4. Imports, 3D-CNN model, and CP helpers

In [4]:
import numpy as np
import scipy.io as sio
import pickle, json, glob, csv
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from scipy.spatial.distance import cdist
from scipy import stats as sstats

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device = {device}')


class CNN3D(nn.Module):
    """3D-CNN for HSI classification (Hamida et al. style, as in SACP paper)."""
    def __init__(self, input_channels, n_classes, patch_size=5):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 20, (3,3,3), padding=0)
        self.pool1 = nn.Conv3d(20, 20, (3,1,1), stride=(2,1,1), padding=(1,0,0))
        self.conv2 = nn.Conv3d(20, 35, (3,3,3), padding=(1,0,0))
        self.pool2 = nn.Conv3d(35, 35, (3,1,1), stride=(2,1,1), padding=(1,0,0))
        self.conv3 = nn.Conv3d(35, 35, (3,1,1), padding=(1,0,0))
        self.conv4 = nn.Conv3d(35, 35, (2,1,1), stride=(2,1,1), padding=(1,0,0))
        self.patch_size = patch_size
        self.input_channels = input_channels
        self.features_size = self._feat_size()
        self.fc = nn.Linear(self.features_size, n_classes)

    def _feat_size(self):
        with torch.no_grad():
            x = torch.zeros((1, 1, self.input_channels, self.patch_size, self.patch_size))
            x = self.pool1(self.conv1(x))
            x = self.pool2(self.conv2(x))
            x = self.conv3(x); x = self.conv4(x)
            return int(np.prod(x.size()[1:]))

    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool1(x)
        x = F.relu(self.conv2(x)); x = self.pool2(x)
        x = F.relu(self.conv3(x)); x = F.relu(self.conv4(x))
        x = x.view(-1, self.features_size)
        return self.fc(x)


# ---- Conformal prediction utilities ----
def aps_scores(probs, labels=None, rng=None):
    rng = rng or np.random
    n, K = probs.shape
    si = np.argsort(-probs, axis=1)
    sp = np.take_along_axis(probs, si, axis=1)
    cs = np.cumsum(sp, axis=1)
    U = rng.uniform(0, 1, size=(n, K))
    ss = cs - sp * U
    scores = np.zeros_like(ss)
    for i in range(n):
        scores[i, si[i]] = ss[i]
    if labels is not None:
        return np.array([scores[i, int(labels[i])] for i in range(n)])
    return scores

def conformal_quantile(scores, alpha):
    n = len(scores)
    return float(np.quantile(scores, np.ceil((n+1)*(1-alpha))/n))

def weighted_quantile(scores, weights, alpha):
    si = np.argsort(scores)
    s = scores[si]; w = weights[si]
    cs = np.cumsum(w); cs = cs / (cs[-1] + 1e-8)
    return float(s[min(np.searchsorted(cs, 1-alpha), len(s)-1)])

def coverage_and_size(psets, labels):
    cov = float(np.mean([int(labels[i]) in s for i, s in enumerate(psets)]))
    sz  = float(np.mean([len(s) for s in psets]))
    return cov, sz

def set_interval_score(psets, true_labels, alpha=0.1):
    total = sum(len(psets[i]) + (2.0/alpha) * (0 if int(true_labels[i]) in psets[i] else 1)
                for i in range(len(psets)))
    return float(total / len(psets))


# ---- SACP: vectorized spatial smoothing over 8-neighbors ----
def sacp_smooth(score_map, h, w, valid_idx, lmd=0.5):
    N = h*w; K = score_map.shape[1]
    sm = score_map.reshape(h, w, K).astype(np.float64)
    vm = np.zeros((h, w), dtype=bool)
    vm.reshape(-1)[valid_idx] = True
    sm_pad = np.pad(sm, ((1,1),(1,1),(0,0)), mode='constant', constant_values=0)
    vm_pad = np.pad(vm, ((1,1),(1,1)), mode='constant', constant_values=False)
    n_sum = np.zeros_like(sm)
    n_cnt = np.zeros((h, w))
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0: continue
            n_sum += sm_pad[1+dr:h+1+dr, 1+dc:w+1+dc, :] * vm_pad[1+dr:h+1+dr, 1+dc:w+1+dc][:,:,None]
            n_cnt += vm_pad[1+dr:h+1+dr, 1+dc:w+1+dc]
    new = sm.copy()
    has = (n_cnt > 0) & vm
    avg = np.zeros_like(sm)
    avg[has] = n_sum[has] / n_cnt[has][:, None]
    new[has] = (1 - lmd) * sm[has] + lmd * avg[has]
    return new.reshape(N, K)


def extract_patches(hsi_chw, indices, patch_size=2):
    d, h, w = hsi_chw.shape
    padded = np.pad(hsi_chw, ((0,0),(patch_size,patch_size),(patch_size,patch_size)), mode='reflect')
    patches = np.zeros((len(indices), 1, d, 2*patch_size+1, 2*patch_size+1), dtype=np.float32)
    for e, idx in enumerate(indices):
        r, c = idx // w, idx % w
        patches[e, 0] = padded[:, r:r+2*patch_size+1, c:c+2*patch_size+1]
    return patches


DATASET_CONFIG = {
    'ip':       ('ip/Indian_pines_corrected.mat', 'indian_pines_corrected',
                 'ip/Indian_pines_gt.mat', 'indian_pines_gt', 16, 200, 'Indian Pines'),
    'pu':       ('pu/PaviaU.mat', 'paviaU',
                 'pu/PaviaU_gt.mat', 'paviaU_gt', 9, 103, 'Pavia University'),
    'sa':       ('sa/Salinas_corrected.mat', 'salinas_corrected',
                 'sa/Salinas_gt.mat', 'salinas_gt', 16, 204, 'Salinas'),
    'ksc':      ('ksc/KSC.mat', 'KSC',
                 'ksc/KSC_gt.mat', 'KSC_gt', 13, 176, 'KSC'),
    'botswana': ('botswana/Botswana.mat', 'Botswana',
                 'botswana/Botswana_gt.mat', 'Botswana_gt', 14, 145, 'Botswana'),
}
print('Helpers loaded. Datasets registered:', list(DATASET_CONFIG.keys()))


device = cuda
Helpers loaded. Datasets registered: ['ip', 'pu', 'sa', 'ksc', 'botswana']


## 5. Core experiment function (single dataset × seed)

Trains 3D-CNN, computes Standard CP / SACP(λ∈{0.3, 0.5, 0.7}) / SACP+GeoCP (CV bw), and saves a **full** pickle checkpoint. If the checkpoint already exists, it is loaded from disk and the expensive training is **skipped**.

In [5]:
def run_experiment(data_name, seed, alpha=0.1, epochs=100, sacp_lmds=(0.3, 0.5, 0.7),
                   bw_list=(3, 5, 7, 10, 15, 20, 30, 50, 100)):
    ckpt_path = f'{CKPT_DIR}/{data_name}_seed{seed}.pkl'
    if os.path.exists(ckpt_path):
        with open(ckpt_path, 'rb') as f:
            return pickle.load(f)

    torch.manual_seed(seed*100 + 42); np.random.seed(seed*100 + 42)
    rng = np.random.RandomState(seed*100 + 42)

    hf, hk, gf, gk, n_classes, bands, nice = DATASET_CONFIG[data_name]
    hsi = sio.loadmat(f'{DATA_DIR}/{hf}')[hk]
    gt  = sio.loadmat(f'{DATA_DIR}/{gf}')[gk]
    h, w, d = hsi.shape; N = h*w

    hsi_norm = hsi.astype(np.float32)
    hsi_norm = (hsi_norm - hsi_norm.mean(axis=(0,1))) / (hsi_norm.max() + 1e-8)
    hsi_chw = hsi_norm.transpose(2, 0, 1)

    y_all = gt.reshape(N)
    labeled_idx = np.where(y_all > 0)[0]
    coords = np.column_stack([np.arange(N)//w, np.arange(N)%w]).astype(np.float32)

    rs = seed*100 + 42
    idx_tr, idx_tmp = train_test_split(range(len(labeled_idx)), train_size=250,
                                        stratify=y_all[labeled_idx]-1, random_state=rs)
    idx_ca, idx_te = train_test_split(idx_tmp, test_size=0.5,
                                       stratify=(y_all[labeled_idx]-1)[idx_tmp], random_state=rs)
    tr_gi, ca_gi, te_gi = labeled_idx[idx_tr], labeled_idx[idx_ca], labeled_idx[idx_te]
    y_tr = y_all[tr_gi]-1; y_ca = y_all[ca_gi]-1; y_te = y_all[te_gi]-1

    # ---- Train 3D-CNN ----
    patch_size = 2
    X_tr = extract_patches(hsi_chw, tr_gi, patch_size)
    X_ca = extract_patches(hsi_chw, ca_gi, patch_size)
    X_te = extract_patches(hsi_chw, te_gi, patch_size)

    model = CNN3D(bands, n_classes, patch_size=5).to(device)
    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                              batch_size=64, shuffle=True)
    opt = optim.Adam(model.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()

    model.eval()
    def get_probs(X):
        loader = DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=256, shuffle=False)
        out = []
        with torch.no_grad():
            for (b,) in loader:
                out.append(torch.softmax(model(b.to(device)), dim=1).cpu().numpy())
        return np.concatenate(out)

    probs_ca = get_probs(X_ca); probs_te = get_probs(X_te)
    pred_te = np.argmax(probs_te, axis=1)
    acc = float(np.mean(pred_te == y_te))

    del X_tr, X_ca, X_te, model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ---- APS base scores ----
    K = n_classes
    cal_all  = aps_scores(probs_ca, rng=rng)
    test_all = aps_scores(probs_te, rng=rng)
    cal_true = aps_scores(probs_ca, y_ca, rng=rng)
    coords_ca = coords[ca_gi]; coords_te = coords[te_gi]

    # ---- 1) Standard CP ----
    q0 = conformal_quantile(cal_true, alpha)
    ps_std = [np.where(test_all[i]<q0)[0].tolist() for i in range(len(idx_te))]
    cov_std, sz_std = coverage_and_size(ps_std, y_te)
    is_std = set_interval_score(ps_std, y_te, alpha)

    # ---- 2) SACP at multiple lambdas ----
    sm = np.zeros((N, K))
    for e, i in enumerate(ca_gi): sm[i] = cal_all[e]
    for e, i in enumerate(te_gi): sm[i] = test_all[e]

    sacp_results = {}
    fused_by_lmd = {}
    for lmd in sacp_lmds:
        fused = sacp_smooth(sm, h, w, np.concatenate([ca_gi, te_gi]), lmd=lmd)
        fused_by_lmd[lmd] = fused
        fcu = np.array([fused[ca_gi[e], int(y_ca[e])] for e in range(len(idx_ca))])
        ftu = np.array([fused[te_gi[e]] for e in range(len(idx_te))])
        qu = conformal_quantile(fcu, alpha)
        ps = [np.where(ftu[i]<qu)[0].tolist() for i in range(len(idx_te))]
        cov, sz = coverage_and_size(ps, y_te)
        sacp_results[float(lmd)] = {
            'cov': cov, 'size': sz, 'is': set_interval_score(ps, y_te, alpha),
            'q': float(qu), 'pred_sets': ps,
        }

    # ---- 3) SACP(λ=0.5) + GeoCP with 5-fold CV for bw ----
    fused05 = fused_by_lmd[0.5]
    fcu05 = np.array([fused05[ca_gi[e], int(y_ca[e])] for e in range(len(idx_ca))])
    ftu05 = np.array([fused05[te_gi[e]] for e in range(len(idx_te))])

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_is = {bw: [] for bw in bw_list}
    for f_tr, f_val in kf.split(np.arange(len(idx_ca))):
        f_ca = ca_gi[f_tr]; f_va = ca_gi[f_val]
        f_fcu = np.array([fused05[f_ca[e], int(y_ca[f_tr[e]])] for e in range(len(f_tr))])
        f_ftu = np.array([fused05[f_va[e]] for e in range(len(f_val))])
        f_yv = y_ca[f_val]
        d_f = cdist(coords[f_va], coords[f_ca])
        for bw in bw_list:
            ps = []
            for i in range(len(f_val)):
                wv = np.exp(-0.5*(d_f[i]/bw)**2); wv /= (wv.sum()+1e-8)
                q = weighted_quantile(f_fcu, wv, alpha)
                ps.append(np.where(f_ftu[i]<q)[0].tolist())
            cv_is[bw].append(set_interval_score(ps, f_yv, alpha))
    best_bw = int(min(bw_list, key=lambda b: np.mean(cv_is[b])))

    # Final eval with CV-selected bw (compute per-pixel q(i))
    dists = cdist(coords_te, coords_ca)
    q_pp = np.zeros(len(idx_te))
    ps_gc = []
    for i in range(len(idx_te)):
        wv = np.exp(-0.5*(dists[i]/best_bw)**2); wv /= (wv.sum()+1e-8)
        q = weighted_quantile(fcu05, wv, alpha)
        q_pp[i] = q
        ps_gc.append(np.where(ftu05[i]<q)[0].tolist())
    cov_gc, sz_gc = coverage_and_size(ps_gc, y_te)
    is_gc = set_interval_score(ps_gc, y_te, alpha)

    # ---- Pack everything ----
    result = {
        'dataset': data_name, 'nice_name': nice, 'seed': int(seed),
        'h': int(h), 'w': int(w), 'n_classes': int(n_classes),
        'bands': int(bands), 'alpha': float(alpha),
        'n_train': len(idx_tr), 'n_calib': len(idx_ca), 'n_test': len(idx_te),
        'accuracy': acc,
        'standard_cp': {'cov': cov_std, 'size': sz_std, 'is': is_std,
                        'q': float(q0), 'pred_sets': ps_std},
        'sacp': sacp_results,
        'sacp_geocp': {'cov': cov_gc, 'size': sz_gc, 'is': is_gc,
                       'bw': best_bw, 'pred_sets': ps_gc,
                       'q_per_pixel': q_pp.tolist(),
                       'cv_is_mean': {int(bw): float(np.mean(v)) for bw, v in cv_is.items()}},
        # indices and labels for later visualization
        'ca_gi': ca_gi.tolist(), 'te_gi': te_gi.tolist(),
        'y_ca': y_ca.tolist(), 'y_te': y_te.tolist(),
        'pred_te': pred_te.tolist(),
    }

    # ---- Save checkpoint IMMEDIATELY ----
    tmp = ckpt_path + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, ckpt_path)  # atomic rename
    return result

print('run_experiment defined')


run_experiment defined


## 6. Run 10 seeds × 5 datasets (resumable)

Each completed seed is pickled to Drive before moving on. Interrupting and re-running this cell is safe.

In [6]:
DATASETS_TO_RUN = ['ip', 'pu', 'sa', 'ksc', 'botswana']
N_SEEDS = 10
EPOCHS  = 100
ALPHA   = 0.1

log_file = f'{RESULTS_DIR}/run_log.txt'
total_runs = len(DATASETS_TO_RUN) * N_SEEDS
done = 0
t_start = time.time()

for ds in DATASETS_TO_RUN:
    print(f'\n{"="*80}\n{DATASET_CONFIG[ds][6]}  ({ds})\n{"="*80}')
    for seed in range(N_SEEDS):
        ckpt = f'{CKPT_DIR}/{ds}_seed{seed}.pkl'
        if os.path.exists(ckpt):
            with open(ckpt, 'rb') as f: r = pickle.load(f)
            done += 1
            print(f'  seed={seed} [cached]  acc={r["accuracy"]:.3f}  '
                  f'StdCP={r["standard_cp"]["is"]:.3f}  '
                  f'SACP(0.5)={r["sacp"][0.5]["is"]:.3f}  '
                  f'SACP+GeoCP={r["sacp_geocp"]["is"]:.3f}  '
                  f'bw={r["sacp_geocp"]["bw"]}  [{done}/{total_runs}]')
            continue
        t0 = time.time()
        try:
            r = run_experiment(ds, seed, alpha=ALPHA, epochs=EPOCHS)
            done += 1
            imp = (r['sacp'][0.5]['is'] - r['sacp_geocp']['is']) / r['sacp'][0.5]['is'] * 100
            msg = (f'  seed={seed}           acc={r["accuracy"]:.3f}  '
                   f'StdCP={r["standard_cp"]["is"]:.3f}  '
                   f'SACP(0.5)={r["sacp"][0.5]["is"]:.3f}  '
                   f'SACP+GeoCP={r["sacp_geocp"]["is"]:.3f} ({imp:+.1f}%)  '
                   f'bw={r["sacp_geocp"]["bw"]}  [{time.time()-t0:.0f}s]  '
                   f'[{done}/{total_runs}]')
            print(msg)
            with open(log_file, 'a') as f: f.write(f'{ds} {msg}\n')
        except Exception as e:
            import traceback
            print(f'  seed={seed} FAILED: {e}')
            traceback.print_exc()
            with open(log_file, 'a') as f: f.write(f'{ds} seed={seed} FAILED: {e}\n')

print(f'\n{"="*80}\nALL DONE: {done}/{total_runs} runs  (total time {(time.time()-t_start)/60:.1f} min)')



Indian Pines  (ip)
  seed=0           acc=0.687  StdCP=4.094  SACP(0.5)=4.039  SACP+GeoCP=3.660 (+9.4%)  bw=7  [20s]  [1/50]
  seed=1           acc=0.678  StdCP=4.403  SACP(0.5)=3.930  SACP+GeoCP=3.600 (+8.4%)  bw=3  [12s]  [2/50]
  seed=2           acc=0.681  StdCP=4.470  SACP(0.5)=4.074  SACP+GeoCP=3.842 (+5.7%)  bw=15  [12s]  [3/50]
  seed=3           acc=0.685  StdCP=4.039  SACP(0.5)=3.809  SACP+GeoCP=3.687 (+3.2%)  bw=15  [12s]  [4/50]
  seed=4           acc=0.702  StdCP=4.118  SACP(0.5)=3.752  SACP+GeoCP=3.450 (+8.1%)  bw=7  [12s]  [5/50]
  seed=5           acc=0.712  StdCP=4.329  SACP(0.5)=3.829  SACP+GeoCP=3.503 (+8.5%)  bw=10  [12s]  [6/50]
  seed=6           acc=0.662  StdCP=4.312  SACP(0.5)=4.123  SACP+GeoCP=3.629 (+12.0%)  bw=5  [12s]  [7/50]
  seed=7           acc=0.695  StdCP=4.189  SACP(0.5)=3.804  SACP+GeoCP=3.611 (+5.1%)  bw=10  [12s]  [8/50]
  seed=8           acc=0.635  StdCP=4.888  SACP(0.5)=4.134  SACP+GeoCP=3.935 (+4.8%)  bw=7  [12s]  [9/50]
  seed=9           ac

## 7. Aggregate → summary.json + per_seed.csv

In [7]:
# Load every cached checkpoint
all_results = {ds: [] for ds in DATASETS_TO_RUN}
for ds in DATASETS_TO_RUN:
    for seed in range(N_SEEDS):
        p = f'{CKPT_DIR}/{ds}_seed{seed}.pkl'
        if os.path.exists(p):
            with open(p, 'rb') as f:
                all_results[ds].append(pickle.load(f))

# ---- summary.json (mean ± std per dataset) ----
def ms(xs):
    arr = np.array(xs, dtype=float)
    return {'mean': float(arr.mean()), 'std': float(arr.std()), 'n': len(arr)}

summary = {}
for ds, rs in all_results.items():
    if not rs: continue
    summary[ds] = {
        'nice_name': rs[0]['nice_name'],
        'n_seeds':   len(rs),
        'accuracy':  ms([r['accuracy'] for r in rs]),
        'standard_cp': {k: ms([r['standard_cp'][k] for r in rs]) for k in ('cov','size','is')},
        'sacp_0.3':    {k: ms([r['sacp'][0.3][k]   for r in rs]) for k in ('cov','size','is')},
        'sacp_0.5':    {k: ms([r['sacp'][0.5][k]   for r in rs]) for k in ('cov','size','is')},
        'sacp_0.7':    {k: ms([r['sacp'][0.7][k]   for r in rs]) for k in ('cov','size','is')},
        'sacp_geocp':  {k: ms([r['sacp_geocp'][k]  for r in rs]) for k in ('cov','size','is')},
        'bw_selected': [int(r['sacp_geocp']['bw']) for r in rs],
    }

with open(f'{RESULTS_DIR}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'[saved] {RESULTS_DIR}/summary.json')

# ---- per_seed.csv (long format) ----
csv_path = f'{RESULTS_DIR}/per_seed.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['dataset', 'seed', 'accuracy',
                'std_cp_cov', 'std_cp_size', 'std_cp_is',
                'sacp03_cov', 'sacp03_size', 'sacp03_is',
                'sacp05_cov', 'sacp05_size', 'sacp05_is',
                'sacp07_cov', 'sacp07_size', 'sacp07_is',
                'geocp_cov', 'geocp_size', 'geocp_is', 'geocp_bw'])
    for ds in DATASETS_TO_RUN:
        for r in all_results[ds]:
            w.writerow([ds, r['seed'], r['accuracy'],
                        r['standard_cp']['cov'], r['standard_cp']['size'], r['standard_cp']['is'],
                        r['sacp'][0.3]['cov'],   r['sacp'][0.3]['size'],   r['sacp'][0.3]['is'],
                        r['sacp'][0.5]['cov'],   r['sacp'][0.5]['size'],   r['sacp'][0.5]['is'],
                        r['sacp'][0.7]['cov'],   r['sacp'][0.7]['size'],   r['sacp'][0.7]['is'],
                        r['sacp_geocp']['cov'],  r['sacp_geocp']['size'],  r['sacp_geocp']['is'],
                        r['sacp_geocp']['bw']])
print(f'[saved] {csv_path}')


[saved] /content/drive/MyDrive/sacp_geocp/results/summary.json
[saved] /content/drive/MyDrive/sacp_geocp/results/per_seed.csv


## 8. Summary table + statistical tests

In [8]:
print(f'{"Dataset":18s} {"N":>3s} {"Acc":>7s}  {"StdCP IS":>14s}  {"SACP(0.5) IS":>15s}  {"SACP+GeoCP IS":>16s}  {"Imp%":>7s}  {"paired p":>10s}')
print('='*115)

stats_out = {}
all_imps, pool_sacp, pool_gc = [], [], []

for ds in DATASETS_TO_RUN:
    rs = all_results[ds]
    if not rs: continue
    accs    = np.array([r['accuracy']         for r in rs])
    std_is  = np.array([r['standard_cp']['is'] for r in rs])
    sacp_is = np.array([r['sacp'][0.5]['is']   for r in rs])
    gc_is   = np.array([r['sacp_geocp']['is']  for r in rs])
    imps    = (sacp_is - gc_is) / sacp_is * 100

    t_stat, p_val = (sstats.ttest_rel(sacp_is, gc_is) if len(rs) > 1 else (0, 1))
    all_imps.extend(imps.tolist()); pool_sacp.extend(sacp_is.tolist()); pool_gc.extend(gc_is.tolist())

    stats_out[ds] = {
        'n_seeds': len(rs), 'mean_imp_pct': float(imps.mean()),
        't_stat': float(t_stat), 'p_value': float(p_val),
    }
    print(f'{DATASET_CONFIG[ds][6]:18s} {len(rs):>3d} {accs.mean():6.3f}  '
          f'{std_is.mean():6.3f}±{std_is.std():.3f}  '
          f'{sacp_is.mean():6.3f}±{sacp_is.std():.3f}  '
          f'{gc_is.mean():6.3f}±{gc_is.std():.3f}  '
          f'{imps.mean():+6.1f}%  {p_val:10.4g}')

pool_sacp = np.array(pool_sacp); pool_gc = np.array(pool_gc)
if len(pool_sacp) > 1:
    t_all, p_all = sstats.ttest_rel(pool_sacp, pool_gc)
    try:    w_stat, p_wil = sstats.wilcoxon(pool_sacp, pool_gc)
    except Exception: w_stat, p_wil = (0, 1)
else:
    t_all, p_all, w_stat, p_wil = 0, 1, 0, 1

print('='*115)
print(f'{"POOLED":18s} {len(pool_sacp):>3d}          '
      f'                {pool_sacp.mean():6.3f}±{pool_sacp.std():.3f}  '
      f'{pool_gc.mean():6.3f}±{pool_gc.std():.3f}  '
      f'{np.mean(all_imps):+6.1f}%')
print(f'\nPooled paired t-test : t={t_all:.3f}, p={p_all:.4g}')
print(f'Wilcoxon signed-rank : W={w_stat:.3f}, p={p_wil:.4g}')
print(f'Positive improvements: {int(np.sum(np.array(all_imps) > 0))}/{len(all_imps)}')

stats_out['pooled'] = {
    'n': int(len(pool_sacp)),
    'mean_imp_pct': float(np.mean(all_imps)),
    't_stat': float(t_all), 'p_value_t': float(p_all),
    'wilcoxon_W': float(w_stat), 'p_value_wilcoxon': float(p_wil),
    'n_positive': int(np.sum(np.array(all_imps) > 0)),
}
with open(f'{RESULTS_DIR}/stats.json', 'w') as f:
    json.dump(stats_out, f, indent=2)
print(f'\n[saved] {RESULTS_DIR}/stats.json')


Dataset              N     Acc        StdCP IS     SACP(0.5) IS     SACP+GeoCP IS     Imp%    paired p
Indian Pines        10  0.688   4.306±0.234   3.920±0.152   3.640±0.145    +7.1%   1.827e-05
Pavia University    10  0.862   3.197±0.091   3.080±0.060   2.986±0.084    +3.0%    0.008655
Salinas             10  0.883   3.142±0.040   3.039±0.046   2.971±0.058    +2.2%      0.0137
KSC                 10  0.811   3.758±0.307   3.435±0.222   3.368±0.213    +1.9%      0.0506
Botswana            10  0.949   2.926±0.171   2.924±0.198   2.903±0.221    +0.7%      0.3802
POOLED              50                           3.280±0.394   3.174±0.326    +3.0%

Pooled paired t-test : t=6.049, p=1.968e-07
Wilcoxon signed-rank : W=113.500, p=4.229e-07
Positive improvements: 41/50

[saved] /content/drive/MyDrive/sacp_geocp/results/stats.json


## 9. LaTeX-ready results table

In [9]:
latex_lines = []
latex_lines.append(r'\begin{tabular}{l|c|ccc|ccc|ccc}')
latex_lines.append(r'\toprule')
latex_lines.append(r' & & \multicolumn{3}{c|}{Standard CP} & \multicolumn{3}{c|}{SACP ($\lambda$=0.5)} & \multicolumn{3}{c}{SACP+GeoCP} \\')
latex_lines.append(r'Dataset & Acc & Cov & Size & IS & Cov & Size & IS & Cov & Size & IS \\')
latex_lines.append(r'\midrule')

PM = r'$\pm$'
def fmt(v, digits=3):
    return f'{v["mean"]:.{digits}f}' + PM + f'{v["std"]:.{digits}f}'

for ds in DATASETS_TO_RUN:
    if ds not in summary: continue
    S = summary[ds]
    row = (f'{S["nice_name"]} & {S["accuracy"]["mean"]:.3f} & '
           f'{fmt(S["standard_cp"]["cov"])} & {fmt(S["standard_cp"]["size"], 2)} & {fmt(S["standard_cp"]["is"], 2)} & '
           f'{fmt(S["sacp_0.5"]["cov"])} & {fmt(S["sacp_0.5"]["size"], 2)} & {fmt(S["sacp_0.5"]["is"], 2)} & '
           f'{fmt(S["sacp_geocp"]["cov"])} & {fmt(S["sacp_geocp"]["size"], 2)} & {fmt(S["sacp_geocp"]["is"], 2)} \\\\')
    latex_lines.append(row)

latex_lines.append(r'\bottomrule')
latex_lines.append(r'\end{tabular}')

latex = '\n'.join(latex_lines).replace('\\\\', '\\\\')
with open(f'{RESULTS_DIR}/results_table.tex', 'w') as f:
    f.write(latex)
print(latex)
print(f'\n[saved] {RESULTS_DIR}/results_table.tex')


\begin{tabular}{l|c|ccc|ccc|ccc}
\toprule
 & & \multicolumn{3}{c|}{Standard CP} & \multicolumn{3}{c|}{SACP ($\lambda$=0.5)} & \multicolumn{3}{c}{SACP+GeoCP} \\
Dataset & Acc & Cov & Size & IS & Cov & Size & IS & Cov & Size & IS \\
\midrule
Indian Pines & 0.688 & 0.903$\pm$0.007 & 2.36$\pm$0.18 & 4.31$\pm$0.23 & 0.901$\pm$0.007 & 1.94$\pm$0.10 & 3.92$\pm$0.15 & 0.916$\pm$0.005 & 1.96$\pm$0.12 & 3.64$\pm$0.15 \\
Pavia University & 0.862 & 0.900$\pm$0.003 & 1.19$\pm$0.05 & 3.20$\pm$0.09 & 0.900$\pm$0.002 & 1.09$\pm$0.05 & 3.08$\pm$0.06 & 0.908$\pm$0.004 & 1.14$\pm$0.04 & 2.99$\pm$0.08 \\
Salinas & 0.883 & 0.902$\pm$0.002 & 1.18$\pm$0.03 & 3.14$\pm$0.04 & 0.901$\pm$0.002 & 1.05$\pm$0.02 & 3.04$\pm$0.05 & 0.908$\pm$0.003 & 1.12$\pm$0.04 & 2.97$\pm$0.06 \\
KSC & 0.811 & 0.893$\pm$0.009 & 1.61$\pm$0.29 & 3.76$\pm$0.31 & 0.896$\pm$0.009 & 1.35$\pm$0.20 & 3.44$\pm$0.22 & 0.906$\pm$0.014 & 1.50$\pm$0.25 & 3.37$\pm$0.21 \\
Botswana & 0.949 & 0.904$\pm$0.010 & 1.00$\pm$0.04 & 2.93$\pm$0.17 & 0.902

## 10. Visualization: 5×4 classification-result grid

Ground Truth · Coverage (green/red) · Pred-set Size · Local q(i) — one row per dataset.

In [10]:
import matplotlib.pyplot as plt

def _bbox_from_gt(gt, pad=3):
    rs, cs = np.where(gt > 0)
    if len(rs) == 0: return 0, gt.shape[0], 0, gt.shape[1]
    r0 = max(0, rs.min() - pad); r1 = min(gt.shape[0], rs.max() + 1 + pad)
    c0 = max(0, cs.min() - pad); c1 = min(gt.shape[1], cs.max() + 1 + pad)
    return r0, r1, c0, c1

def plot_viz_grid(save_path, seed=0):
    rows = []
    for ds in DATASETS_TO_RUN:
        p = f'{CKPT_DIR}/{ds}_seed{seed}.pkl'
        if not os.path.exists(p):
            print(f'[skip] {ds} seed={seed} has no checkpoint'); continue
        with open(p, 'rb') as f: r = pickle.load(f)
        hf, hk, gf, gk, _, _, _ = DATASET_CONFIG[ds]
        r['gt'] = sio.loadmat(f'{DATA_DIR}/{gf}')[gk]
        rows.append(r)
    if not rows:
        print('nothing to plot'); return

    n = len(rows)
    fig, axes = plt.subplots(n, 4, figsize=(16, 3.4*n))
    if n == 1: axes = axes[None, :]

    for row, R in enumerate(rows):
        h, w = R['h'], R['w']; gt = R['gt']; n_cls = R['n_classes']
        te_gi = np.array(R['te_gi']); y_te = np.array(R['y_te'])
        ps_gc = R['sacp_geocp']['pred_sets']
        q_pp  = np.array(R['sacp_geocp']['q_per_pixel'])
        rotate = (h > 2*w)
        r0, r1, c0, c1 = _bbox_from_gt(gt)
        te_rc = [(int(gi // w), int(gi % w)) for gi in te_gi]
        area_per_lab = (h*w) / max(1, int((gt > 0).sum()))
        radius = 1 if area_per_lab < 3 else (2 if area_per_lab < 10 else 3)
        _rot = (np.rot90 if rotate else (lambda a: a))

        # --- Col 0: GT ---
        ax = axes[row, 0]
        cmap = plt.get_cmap('tab20', max(n_cls, 20))
        gt_m = np.ma.masked_where(gt == 0, gt.astype(float))[r0:r1, c0:c1]
        ax.imshow(_rot(gt_m), cmap=cmap, vmin=1, vmax=n_cls, interpolation='nearest')
        ax.set_title(f"{R['nice_name']} — GT ({n_cls} cls, acc={R['accuracy']:.2f})", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

        # --- Col 1: Coverage ---
        ax = axes[row, 1]
        rgb = np.ones((h, w, 3))
        gt_bg = (gt > 0).astype(float)[:, :, None]
        rgb = rgb * (0.15 + 0.85*gt_bg) + 0.85*(1-gt_bg)
        for i, (r, c) in enumerate(te_rc):
            is_cov = int(y_te[i]) in ps_gc[i]
            ra, rb = max(0, r-radius), min(h, r+radius+1)
            ca, cb = max(0, c-radius), min(w, c+radius+1)
            rgb[ra:rb, ca:cb] = [0.2, 0.75, 0.25] if is_cov else [0.95, 0.15, 0.15]
        ax.imshow(_rot(rgb[r0:r1, c0:c1]), interpolation='nearest')
        ax.set_title(f"Coverage = {R['sacp_geocp']['cov']:.3f}", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

        # --- Col 2: Size ---
        ax = axes[row, 2]
        sz_map = np.full((h, w), np.nan)
        for i, (r, c) in enumerate(te_rc):
            sz = len(ps_gc[i])
            ra, rb = max(0, r-radius), min(h, r+radius+1)
            ca, cb = max(0, c-radius), min(w, c+radius+1)
            sz_map[ra:rb, ca:cb] = sz
        vmax = min(n_cls, max(3, int(np.nanmax(sz_map)) if not np.all(np.isnan(sz_map)) else 3))
        im = ax.imshow(_rot(sz_map[r0:r1, c0:c1]), cmap='viridis', vmin=1, vmax=vmax, interpolation='nearest')
        ax.set_title(f"Pred-set Size (mean={R['sacp_geocp']['size']:.2f})", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

        # --- Col 3: Local q(i) ---
        ax = axes[row, 3]
        q_map = np.full((h, w), np.nan)
        for i, (r, c) in enumerate(te_rc):
            q = q_pp[i]
            ra, rb = max(0, r-radius), min(h, r+radius+1)
            ca, cb = max(0, c-radius), min(w, c+radius+1)
            q_map[ra:rb, ca:cb] = q
        im = ax.imshow(_rot(q_map[r0:r1, c0:c1]), cmap='plasma', interpolation='nearest')
        ax.set_title(f"Local q(i)  (bw={R['sacp_geocp']['bw']})", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'[saved] {save_path}')

plot_viz_grid(f'{FIG_DIR}/fig_all_datasets_results.png', seed=0)


[saved] /content/drive/MyDrive/sacp_geocp/figures/fig_all_datasets_results.png


## 11. Package & download results

Zips `results/` + `figures/` so you can download them off Colab in one shot.

In [11]:
import shutil

results_zip  = '/content/sacp_geocp_results.zip'
figures_zip  = '/content/sacp_geocp_figures.zip'
checkpts_zip = '/content/sacp_geocp_checkpoints.zip'

shutil.make_archive(results_zip[:-4],  'zip', WORK_DIR, 'results')
shutil.make_archive(figures_zip[:-4],  'zip', WORK_DIR, 'figures')
shutil.make_archive(checkpts_zip[:-4], 'zip', WORK_DIR, 'checkpoints')

for z in (results_zip, figures_zip, checkpts_zip):
    print(f'  {z}  ({os.path.getsize(z)//1024} KB)')

# Prompt auto-download when running on Colab
try:
    from google.colab import files
    files.download(results_zip)
    files.download(figures_zip)
except Exception as e:
    print(f'\n(auto-download skipped: {e})')
    print(f'Files also live at: {WORK_DIR}/results and {WORK_DIR}/figures on Drive')


  /content/sacp_geocp_results.zip  (10 KB)
  /content/sacp_geocp_figures.zip  (339 KB)
  /content/sacp_geocp_checkpoints.zip  (8630 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>